<a href="https://colab.research.google.com/github/GabAsencios/FocusGuard/blob/main/FocusGuard_CamDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this notebook i will train a YoloV8 model with COCO8 datasets for phone and person detection.

In [1]:
# Setup
!pip install ultralytics
import os
from ultralytics import YOLO

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 30.5 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [2]:
def prepare_webcam_model(model_variant='yolov8n.pt'):
    """
    Loads a pretrained YOLOv8 model and prepares it for export.

    Args:
        model_variant (str): The YOLOv8 model version (n, s, m, l, or x).

    Returns:
        YOLO: The loaded ultralytics YOLO model.

    Example:
        model = prepare_webcam_model('yolov8n.pt')
    """
    # Load pretrained COCO weights as per proposal
    model = YOLO(model_variant)

    # Optional: If you collect a custom dataset of 'cellphone' vs 'not-cellphone'
    # to show training competence[cite: 270], uncomment the line below:
    # model.train(data='custom_data.yaml', epochs=10, imgsz=640)

    # Save/Export the model
    model.export(format='torchscript')
    return model

In [3]:
# 3. Prepare Webcam Component [cite: 209]
# Run the preparation
model = prepare_webcam_model()

Ultralytics 8.4.38 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs

PyTorch: starting from 'yolov8n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (6.2 MB)

TorchScript: starting export with torch 2.10.0+cpu...
TorchScript: export success ✅ 5.2s, saved as 'yolov8n.torchscript' (12.5 MB)

Export complete (6.8s)
Results saved to /content
Predict:         yolo predict task=detect model=yolov8n.torchscript imgsz=640 
Validate:        yolo val task=detect model=yolov8n.torchscript imgsz=640 data=coco.yaml  
Visualize:       https://netron.app


In [4]:
# Download the weights (best.pt or yolov8n.pt) for local use
from google.colab import files
files.download('yolov8n.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 4. Evaluate the model with testing metrics

To evaluate the model, we can use the `model.val()` method. This will run the validation process on a specified dataset (COCO in this case) and output various performance metrics.

In [5]:
# Evaluate the model's performance on the COCO dataset
metrics = model.val(data='coco.yaml')

# Print some key metrics
print(f"Map50-95: {metrics.box.map}")
print(f"Map50: {metrics.box.map50}")
print(f"Map75: {metrics.box.map75}")

# You can also access individual class metrics if needed
# for i, class_name in enumerate(metrics.names):
#     print(f"Class {class_name} mAP: {metrics.results_dict[f'metrics/mAP50-95(B)/{class_name}']}")

Ultralytics 8.4.38 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs

WARNING ⚠️ Dataset 'coco.yaml' images not found, missing path '/content/datasets/coco/val2017.txt'
Unzipping /content/datasets/coco2017labels-segments.zip to /content/datasets/coco...: 100% ━━━━━━━━━━━━ 122232/122232 3.9Kfiles/s 31.5s
Unzipping /content/datasets/coco/images/test2017.zip to /content/datasets/coco/images/test2017...: 100% ━━━━━━━━━━━━ 40671/40671 420.1files/s 1:37
Unzipping /content/datasets/coco/images/train2017.zip to /content/datasets/coco/images/train2017...: 100% ━━━━━━━━━━━━ 118288/118288 562.0files/s 3:30
Dataset download success ✅ (725.5s), saved to /content/datasets

val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 60.3±28.3 MB/s, size: 203.7 KB)
val: Scanning /content/datasets/coco/labels/val2017... 4952 images, 48 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5000/5000 266.7it/s 18.7s
val: New cache